Explanation

Daily grain: Each row is a single metric for a single day.

Customer growth: (new_customers / previous_day_total_customers) * 100 → null-safe.

Repeat customer rate: % of customers with >1 order in last 30 days.

Top 10 products: Calculates top 10 product sales as a % of total sales per day.

Profit margin & avg_discount: Derived directly from mart_sales_daily.

Long format: Perfect for dashboards — BI tools can pivot or filter easily.

In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_gold_schema, get_kpi_schema, get_mart_schema, get_semantic_layer_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
mart_schema = get_mart_schema()
semantic_layer_schema = get_semantic_layer_schema()  # e.g., dev_platform



# -------------------------------
# Create kpi View (Semantic Layer for Analysts)
# -------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{semantic_layer_schema}.metrics_business_kpi AS
-- Step 1: Daily Sales Metrics
WITH daily_sales AS (
    SELECT
        order_date AS metric_date,
        SUM(total_sales) AS total_sales,
        SUM(total_profit) AS total_profit,
        CASE WHEN SUM(total_sales) = 0 THEN NULL ELSE ROUND(SUM(total_profit)/SUM(total_sales),4) END AS profit_margin,
        AVG(avg_discount) AS avg_discount
    FROM {catalog}.{mart_schema}.mart_sales_daily
    GROUP BY order_date
),

-- Step 2: Customer Daily Metrics from mart
customer_daily AS (
    SELECT
        s.order_date AS metric_date,
        COUNT(DISTINCT c.customer_id) AS total_customers,
        COUNT(DISTINCT CASE WHEN s.order_date = CURRENT_DATE() THEN c.customer_id END) AS new_customers
    FROM {catalog}.{mart_schema}.mart_customer_360 c
    LEFT JOIN {catalog}.{mart_schema}.mart_sales_daily s
        ON c.customer_id = s.customer_id
    GROUP BY s.order_date
),

-- Step 3: Repeat Customer Metrics (30-day repeat) from mart
repeat_customers AS (
    SELECT
        s.order_date AS metric_date,
        COUNT(DISTINCT c.customer_id) AS repeat_customers_30d
    FROM {catalog}.{mart_schema}.mart_customer_360 c
    LEFT JOIN {catalog}.{mart_schema}.mart_sales_daily s
        ON c.customer_id = s.customer_id
    WHERE s.order_date >= DATE_SUB(CURRENT_DATE(), 30) 
      AND c.total_orders > 1  -- total_orders in last 30 days ideally
    GROUP BY s.order_date
),

-- Step 4: Top 10 Product Sales % per Day
top_products AS (
    SELECT
        order_date AS metric_date,
        SUM(total_sales) AS total_sales,
        SUM(total_sales) FILTER (WHERE rank <= 10) AS top_10_sales
    FROM (
        SELECT
            order_date,
            product_id,
            total_sales,
            ROW_NUMBER() OVER (PARTITION BY order_date ORDER BY total_sales DESC) AS rank
        FROM {catalog}.{mart_schema}.mart_sales_daily
    ) ranked
    GROUP BY order_date
)

-- Step 5: Combine all metrics in long format
SELECT metric_date, 'total_sales' AS metric_name, total_sales AS metric_value FROM daily_sales
UNION ALL
SELECT metric_date, 'total_profit', total_profit FROM daily_sales
UNION ALL
SELECT metric_date, 'profit_margin', profit_margin FROM daily_sales
UNION ALL
SELECT metric_date, 'avg_discount', avg_discount FROM daily_sales
UNION ALL
SELECT cd.metric_date, 'customer_growth_rate' AS metric_name,
       CASE 
           WHEN LAG(cd.total_customers) OVER (ORDER BY cd.metric_date) = 0 THEN NULL
           ELSE ROUND((cd.new_customers * 100.0) / LAG(cd.total_customers) OVER (ORDER BY cd.metric_date),2)
       END AS metric_value
FROM customer_daily cd
UNION ALL
SELECT cd.metric_date, 'repeat_customer_rate' AS metric_name,
       CASE 
           WHEN cd.total_customers = 0 THEN NULL
           ELSE ROUND((rc.repeat_customers_30d * 100.0) / cd.total_customers,2)
       END AS metric_value
FROM repeat_customers rc
JOIN customer_daily cd ON rc.metric_date = cd.metric_date
UNION ALL
SELECT tp.metric_date, 'top_10_product_sales_pct' AS metric_name,
       CASE WHEN tp.total_sales = 0 THEN NULL ELSE ROUND(tp.top_10_sales * 100.0 / tp.total_sales,2) END AS metric_value
FROM top_products tp;
""")
